# Python 进阶实操教程：标准库工程化实践

你会练习文件系统操作、路径管理、JSON/CSV 数据处理、SQLite 数据库、日志、命令行参数、正则表达式、日期时间、压缩包、哈希、并发、异步、子进程、单元测试和一个综合项目。

# 环境与目录准备

进阶实操中建议把文件统一放在一个工作目录中，避免把临时文件散落在 Notebook 所在目录。

In [1]:
from pathlib import Path
import sys

BASE_DIR = Path("python_stdlib_practice")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
LOG_DIR = BASE_DIR / "logs"

for path in [BASE_DIR, DATA_DIR, OUTPUT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("工作目录:", BASE_DIR.resolve())

Python: 3.11.8 (main, Feb 26 2024, 21:39:34) [GCC 11.2.0]
工作目录: /home/lyz/Work/python_stdlib_practice


# pathlib：现代路径管理

`pathlib` 是现代 Python 推荐的路径处理方式，相比字符串拼接更安全、更清晰，也更适合跨平台项目。

In [2]:
from pathlib import Path

file_path = DATA_DIR / "hello.txt"

file_path.write_text("Hello pathlib\n这是一个路径管理示例", encoding="utf-8")

print("文件是否存在:", file_path.exists())
print("文件名:", file_path.name)
print("后缀:", file_path.suffix)
print("父目录:", file_path.parent)
print("绝对路径:", file_path.resolve())
print("文件内容:")
print(file_path.read_text(encoding="utf-8"))

文件是否存在: True
文件名: hello.txt
后缀: .txt
父目录: python_stdlib_practice/data
绝对路径: /home/lyz/Work/python_stdlib_practice/data/hello.txt
文件内容:
Hello pathlib
这是一个路径管理示例


# os、shutil 与 glob：文件系统批处理

`os` 适合处理环境变量、进程目录等底层操作，`shutil` 适合复制、移动、删除文件，`glob` 适合按模式搜索文件。

In [3]:
import os
import shutil
import glob

for i in range(5):
    path = DATA_DIR / f"sample_{i}.txt"
    path.write_text(f"这是第 {i} 个文件\n", encoding="utf-8")

backup_dir = OUTPUT_DIR / "backup"
backup_dir.mkdir(exist_ok=True)

for file in DATA_DIR.glob("sample_*.txt"):
    shutil.copy(file, backup_dir / file.name)

print("当前工作目录:", os.getcwd())
print("环境变量 PATH 是否存在:", "PATH" in os.environ)
print("glob 搜索结果:")
for file in glob.glob(str(backup_dir / "*.txt")):
    print(file)

当前工作目录: /home/lyz/Work
环境变量 PATH 是否存在: True
glob 搜索结果:
python_stdlib_practice/output/backup/sample_0.txt
python_stdlib_practice/output/backup/sample_3.txt
python_stdlib_practice/output/backup/sample_1.txt
python_stdlib_practice/output/backup/sample_4.txt
python_stdlib_practice/output/backup/sample_2.txt


# tempfile：安全创建临时文件

在真实项目中，不建议随便创建临时文件名，因为容易冲突。`tempfile` 可以安全地创建临时目录和临时文件。

In [4]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_path = Path(tmpdir)
    temp_file = tmp_path / "temp.txt"
    temp_file.write_text("临时文件内容", encoding="utf-8")
    print("临时目录:", tmp_path)
    print("临时文件存在:", temp_file.exists())
    print("临时文件内容:", temp_file.read_text(encoding="utf-8"))

print("退出 with 后临时目录会被自动清理")

临时目录: /tmp/tmp8yfybzdr
临时文件存在: True
临时文件内容: 临时文件内容
退出 with 后临时目录会被自动清理


# json：结构化数据读写

`json` 常用于配置文件、接口数据、模型输出和轻量级数据存储。

In [5]:
import json

users = [
    {"id": 1, "name": "小明", "age": 20, "skills": ["Python", "SQL"]},
    {"id": 2, "name": "小红", "age": 22, "skills": ["Excel", "Python"]},
    {"id": 3, "name": "小刚", "age": 19, "skills": ["Linux", "Git"]}
]

json_path = DATA_DIR / "users.json"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

with json_path.open("r", encoding="utf-8") as f:
    loaded_users = json.load(f)

print(loaded_users)
print("第一个用户:", loaded_users[0])

[{'id': 1, 'name': '小明', 'age': 20, 'skills': ['Python', 'SQL']}, {'id': 2, 'name': '小红', 'age': 22, 'skills': ['Excel', 'Python']}, {'id': 3, 'name': '小刚', 'age': 19, 'skills': ['Linux', 'Git']}]
第一个用户: {'id': 1, 'name': '小明', 'age': 20, 'skills': ['Python', 'SQL']}


# csv：表格数据读写

`csv` 适合处理简单表格数据。在数据分析、爬虫结果保存、日志导出中很常见。

In [6]:
import csv

orders = [
    {"order_id": "A001", "user": "小明", "amount": 199.0},
    {"order_id": "A002", "user": "小红", "amount": 299.5},
    {"order_id": "A003", "user": "小刚", "amount": 88.8}
]

csv_path = DATA_DIR / "orders.csv"

with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_id", "user", "amount"])
    writer.writeheader()
    writer.writerows(orders)

with csv_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(rows)
total = sum(float(row["amount"]) for row in rows)
print("总金额:", total)

[{'order_id': 'A001', 'user': '小明', 'amount': '199.0'}, {'order_id': 'A002', 'user': '小红', 'amount': '299.5'}, {'order_id': 'A003', 'user': '小刚', 'amount': '88.8'}]
总金额: 587.3


# configparser：读取配置文件

复杂项目中，不应该把路径、数据库名、参数阈值写死在代码中，可以放到配置文件里。

In [7]:
import configparser

config = configparser.ConfigParser()
config["app"] = {
    "name": "stdlib-practice",
    "debug": "true"
}
config["database"] = {
    "path": str(DATA_DIR / "app.db")
}

config_path = BASE_DIR / "config.ini"

with config_path.open("w", encoding="utf-8") as f:
    config.write(f)

loaded_config = configparser.ConfigParser()
loaded_config.read(config_path, encoding="utf-8")

print("应用名:", loaded_config["app"]["name"])
print("是否调试:", loaded_config["app"].getboolean("debug"))
print("数据库路径:", loaded_config["database"]["path"])

应用名: stdlib-practice
是否调试: True
数据库路径: python_stdlib_practice/data/app.db


# datetime 与 zoneinfo：日期时间处理

真实业务中经常需要处理时间戳、日期格式、时区、时间差等问题。

In [8]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

now = datetime.now()
utc_now = datetime.now(timezone.utc)
berlin_now = datetime.now(ZoneInfo("Europe/Berlin"))
shanghai_now = datetime.now(ZoneInfo("Asia/Shanghai"))

print("本地时间:", now)
print("UTC 时间:", utc_now)
print("柏林时间:", berlin_now)
print("上海时间:", shanghai_now)

deadline = now + timedelta(days=7, hours=3)
print("截止时间:", deadline.strftime("%Y-%m-%d %H:%M:%S"))

date_text = "2026-07-06 14:30:00"
parsed = datetime.strptime(date_text, "%Y-%m-%d %H:%M:%S")
print("解析后的时间:", parsed)

本地时间: 2026-07-07 16:51:24.574847
UTC 时间: 2026-07-07 08:51:24.574926+00:00
柏林时间: 2026-07-07 10:51:24.672782+02:00
上海时间: 2026-07-07 16:51:24.735774+08:00
截止时间: 2026-07-14 19:51:24
解析后的时间: 2026-07-06 14:30:00


# re：正则表达式文本处理

正则表达式适合从文本中提取邮箱、手机号、日期、编号等结构化信息。

In [9]:
import re

text = """
联系人：Alice，邮箱 alice@example.com，订单号 A-2026-001
联系人：Bob，邮箱 bob@test.org，订单号 A-2026-002
错误邮箱：not-an-email
"""

email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
order_pattern = r"A-\d{4}-\d{3}"

emails = re.findall(email_pattern, text)
orders = re.findall(order_pattern, text)

print("邮箱:", emails)
print("订单号:", orders)

masked = re.sub(email_pattern, "[EMAIL]", text)
print(masked)

邮箱: ['alice@example.com', 'bob@test.org']
订单号: ['A-2026-001', 'A-2026-002']

联系人：Alice，邮箱 [EMAIL]，订单号 A-2026-001
联系人：Bob，邮箱 [EMAIL]，订单号 A-2026-002
错误邮箱：not-an-email



# collections：更高效的数据容器

`collections` 提供了 `Counter`、`defaultdict`、`deque` 等实用数据结构。

In [10]:
from collections import Counter, defaultdict, deque

words = "python data python ai data python".split()
counter = Counter(words)
print("词频:", counter)
print("最常见:", counter.most_common(2))

groups = defaultdict(list)
students = [
    ("一班", "小明"),
    ("一班", "小红"),
    ("二班", "小刚")
]

for class_name, student_name in students:
    groups[class_name].append(student_name)

print("分组:", dict(groups))

queue = deque(["task1", "task2"])
queue.append("task3")
print("出队:", queue.popleft())
print("剩余队列:", queue)

词频: Counter({'python': 3, 'data': 2, 'ai': 1})
最常见: [('python', 3), ('data', 2)]
分组: {'一班': ['小明', '小红'], '二班': ['小刚']}
出队: task1
剩余队列: deque(['task2', 'task3'])


# itertools 与 functools：函数式工具

这些工具常用于组合迭代、批处理、缓存和函数封装。

In [11]:
import itertools
import functools

items = ["A", "B", "C"]
print("两两组合:", list(itertools.combinations(items, 2)))
print("笛卡尔积:", list(itertools.product(["红", "蓝"], ["S", "M"])))

numbers = [1, 2, 3, 4, 5]
total = functools.reduce(lambda a, b: a + b, numbers)
print("reduce 求和:", total)

@functools.lru_cache(maxsize=128)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print("fib(30):", fib(30))
print("缓存信息:", fib.cache_info())

两两组合: [('A', 'B'), ('A', 'C'), ('B', 'C')]
笛卡尔积: [('红', 'S'), ('红', 'M'), ('蓝', 'S'), ('蓝', 'M')]
reduce 求和: 15
fib(30): 832040
缓存信息: CacheInfo(hits=28, misses=31, maxsize=128, currsize=31)


# dataclasses、enum 与 typing：更清晰的数据建模

`dataclasses` 可以减少样板代码，`enum` 可以表示有限状态，`typing` 可以增强代码可读性。

In [12]:
from dataclasses import dataclass, field
from enum import Enum
from typing import List

class OrderStatus(Enum):
    CREATED = "created"
    PAID = "paid"
    CANCELLED = "cancelled"

@dataclass
class Order:
    order_id: str
    user: str
    amount: float
    status: OrderStatus = OrderStatus.CREATED
    tags: List[str] = field(default_factory=list)

    def pay(self):
        self.status = OrderStatus.PAID

order = Order(order_id="A001", user="小明", amount=199.0, tags=["new_user"])
print(order)
order.pay()
print(order.status)

Order(order_id='A001', user='小明', amount=199.0, status=<OrderStatus.CREATED: 'created'>, tags=['new_user'])
OrderStatus.PAID


# logging：工程化日志

真实项目中不要只依赖 `print`，日志可以区分级别、保存到文件，并便于排查问题。

In [13]:
import logging

log_path = LOG_DIR / "app.log"

logger = logging.getLogger("practice_logger")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")

file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("程序启动")
logger.warning("这是一个警告示例")
logger.error("这是一个错误示例")

print("日志文件内容:")
print(log_path.read_text(encoding="utf-8"))

2026-07-07 16:51:25,095 | INFO | practice_logger | 程序启动
2026-07-07 16:51:25,096 | WARNING | practice_logger | 这是一个警告示例
2026-07-07 16:51:25,097 | ERROR | practice_logger | 这是一个错误示例


日志文件内容:
2026-07-07 16:51:25,095 | INFO | practice_logger | 程序启动
2026-07-07 16:51:25,096 | WARNING | practice_logger | 这是一个警告示例
2026-07-07 16:51:25,097 | ERROR | practice_logger | 这是一个错误示例



# sqlite3：轻量级数据库

`sqlite3` 是 Python 标准库自带的数据库，适合本地数据存储、小型应用、原型系统和测试环境。

In [14]:
import sqlite3

db_path = DATA_DIR / "shop.db"

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user TEXT NOT NULL,
        product TEXT NOT NULL,
        amount REAL NOT NULL,
        created_at TEXT NOT NULL
    )
    """)
    conn.commit()

print("数据库已创建:", db_path)

数据库已创建: python_stdlib_practice/data/shop.db


In [15]:
from datetime import datetime

new_orders = [
    ("小明", "键盘", 199.0, datetime.now().isoformat()),
    ("小红", "鼠标", 59.0, datetime.now().isoformat()),
    ("小刚", "显示器", 899.0, datetime.now().isoformat())
]

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.executemany(
        "INSERT INTO orders (user, product, amount, created_at) VALUES (?, ?, ?, ?)",
        new_orders
    )
    conn.commit()

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT user, product, amount FROM orders WHERE amount >= ?", (100,))
    result = cursor.fetchall()

print(result)

[('小明', '键盘', 199.0), ('小刚', '显示器', 899.0)]


# urllib.request：标准库网络请求

不安装第三方库时，也可以用 `urllib.request` 发送简单 HTTP 请求。

In [16]:
from urllib.request import urlopen, Request
import json

url = "https://httpbin.org/json"
request = Request(url, headers={"User-Agent": "Python-stdlib-practice"})

try:
    with urlopen(request, timeout=10) as response:
        data = json.loads(response.read().decode("utf-8"))
    print(data)
except Exception as e:
    print("网络请求失败，可能是当前环境无法访问外网:", repr(e))

网络请求失败，可能是当前环境无法访问外网: <HTTPError 503: 'Service Temporarily Unavailable'>


# http.server：快速启动本地静态文件服务器

`http.server` 可以快速把一个目录作为静态网站目录暴露出来，适合本地测试。

In [17]:
import http.server
import socketserver
import threading
from functools import partial

site_dir = OUTPUT_DIR / "site"
site_dir.mkdir(exist_ok=True)
(site_dir / "index.html").write_text("<h1>Hello http.server</h1>", encoding="utf-8")

PORT = 8000

Handler = partial(http.server.SimpleHTTPRequestHandler, directory=str(site_dir))

def run_server():
    with socketserver.TCPServer(("", PORT), Handler) as httpd:
        httpd.timeout = 1
        print(f"本地服务启动：http://localhost:{PORT}")
        httpd.handle_request()

server_thread = threading.Thread(target=run_server)
server_thread.start()
server_thread.join()

print("示例服务已处理一次请求或超时退出")

本地服务启动：http://localhost:8000
示例服务已处理一次请求或超时退出


# subprocess：调用外部命令

`subprocess` 可以让 Python 调用系统命令、脚本、编译器或其他程序。

In [18]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-c", "print('hello from subprocess')"],
    capture_output=True,
    text=True
)

print("返回码:", result.returncode)
print("标准输出:", result.stdout)
print("标准错误:", result.stderr)

返回码: 0
标准输出: hello from subprocess

标准错误: 


# threading 与 queue：线程和任务队列

线程适合处理 I/O 密集型任务，例如文件读写、网络请求、等待外部服务返回。

In [19]:
import threading
import queue
import time

task_queue = queue.Queue()
result_queue = queue.Queue()

for i in range(5):
    task_queue.put(i)

def worker():
    while True:
        try:
            task = task_queue.get_nowait()
        except queue.Empty:
            break
        time.sleep(0.2)
        result_queue.put((task, task * task))
        task_queue.task_done()

threads = [threading.Thread(target=worker) for _ in range(3)]

for t in threads:
    t.start()

for t in threads:
    t.join()

results = []
while not result_queue.empty():
    results.append(result_queue.get())

print(sorted(results))

[(0, 0), (1, 1), (2, 4), (3, 9), (4, 16)]


# concurrent.futures：更简单的线程池和进程池

`ThreadPoolExecutor` 适合 I/O 密集型任务，`ProcessPoolExecutor` 适合 CPU 密集型任务。

In [20]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fake_download(task_id):
    time.sleep(0.3)
    return f"任务 {task_id} 完成"

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(fake_download, i) for i in range(6)]

    for future in as_completed(futures):
        print(future.result())

任务 0 完成
任务 1 完成
任务 2 完成
任务 3 完成
任务 4 完成
任务 5 完成


In [21]:
from concurrent.futures import ProcessPoolExecutor
import math

def cpu_task(n):
    total = 0
    for i in range(1, n):
        total += math.sqrt(i)
    return total

try:
    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(cpu_task, [10000, 20000, 30000]))
    print(results)
except Exception as e:
    print("进程池执行失败，可能是 Notebook 环境限制:", repr(e))

[666616.4591971082, 1885547.164894411, 3464014.8049517325]


# asyncio：异步编程基础

`asyncio` 适合大量等待型任务，例如并发请求接口、等待文件或网络 I/O。注意：异步不是多线程，它通过事件循环在等待期间切换任务。

In [22]:
import asyncio
import time

async def async_task(name, delay):
    print(f"{name} 开始")
    await asyncio.sleep(delay)
    print(f"{name} 完成")
    return name

start = time.perf_counter()

results = await asyncio.gather(
    async_task("任务A", 1),
    async_task("任务B", 1.5),
    async_task("任务C", 0.5)
)

end = time.perf_counter()

print("结果:", results)
print("耗时:", round(end - start, 2), "秒")

任务A 开始
任务B 开始
任务C 开始
任务C 完成
任务A 完成
任务B 完成
结果: ['任务A', '任务B', '任务C']
耗时: 1.5 秒


# asyncio.Queue：异步生产者消费者

异步队列适合构建任务流水线，例如抓取网页、清洗数据、写入数据库。

In [23]:
import asyncio
import random

async def producer(queue):
    for i in range(5):
        await asyncio.sleep(0.1)
        item = f"item-{i}"
        await queue.put(item)
        print("生产:", item)

async def consumer(queue, name):
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            break
        await asyncio.sleep(random.uniform(0.1, 0.3))
        print(f"{name} 消费:", item)
        queue.task_done()

queue = asyncio.Queue()

consumers = [
    asyncio.create_task(consumer(queue, "消费者A")),
    asyncio.create_task(consumer(queue, "消费者B"))
]

await producer(queue)

await queue.put(None)
await queue.put(None)

await queue.join()
await asyncio.gather(*consumers)

print("异步队列处理完成")

生产: item-0
生产: item-1
消费者A 消费: item-0
生产: item-2
消费者B 消费: item-1
生产: item-3
消费者A 消费: item-2
生产: item-4
消费者B 消费: item-3
消费者A 消费: item-4
异步队列处理完成


# contextlib：上下文管理器

上下文管理器可以统一管理资源的申请和释放，例如文件、连接、锁、计时器。

In [24]:
from contextlib import contextmanager
import time

@contextmanager
def timer(name):
    start = time.perf_counter()
    print(f"{name} 开始")
    try:
        yield
    finally:
        end = time.perf_counter()
        print(f"{name} 结束，耗时 {end - start:.4f} 秒")

with timer("模拟任务"):
    total = sum(range(1000000))

print(total)

模拟任务 开始
模拟任务 结束，耗时 0.0429 秒
499999500000


# zipfile、tarfile 与 gzip：压缩文件处理

标准库可以直接创建和读取常见压缩格式。

In [25]:
import zipfile
import gzip

zip_path = OUTPUT_DIR / "backup.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in DATA_DIR.glob("*.txt"):
        zf.write(file, arcname=file.name)

print("zip 文件:", zip_path)

extract_dir = OUTPUT_DIR / "extracted"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

print("解压结果:", list(extract_dir.iterdir()))

gzip_path = OUTPUT_DIR / "data.txt.gz"
with gzip.open(gzip_path, "wt", encoding="utf-8") as f:
    f.write("这是一段 gzip 压缩文本")

with gzip.open(gzip_path, "rt", encoding="utf-8") as f:
    print(f.read())

zip 文件: python_stdlib_practice/output/backup.zip
解压结果: [PosixPath('python_stdlib_practice/output/extracted/sample_0.txt'), PosixPath('python_stdlib_practice/output/extracted/sample_3.txt'), PosixPath('python_stdlib_practice/output/extracted/hello.txt'), PosixPath('python_stdlib_practice/output/extracted/sample_1.txt'), PosixPath('python_stdlib_practice/output/extracted/sample_4.txt'), PosixPath('python_stdlib_practice/output/extracted/sample_2.txt')]
这是一段 gzip 压缩文本


# hashlib、hmac、base64、uuid 与 secrets：常用安全工具

这些库常用于文件校验、签名、编码、唯一 ID 和安全随机数生成。

In [26]:
import hashlib
import hmac
import base64
import uuid
import secrets

content = b"hello python"

md5_value = hashlib.md5(content).hexdigest()
sha256_value = hashlib.sha256(content).hexdigest()

print("MD5:", md5_value)
print("SHA256:", sha256_value)

secret_key = b"my-secret"
signature = hmac.new(secret_key, content, hashlib.sha256).hexdigest()
print("HMAC 签名:", signature)

encoded = base64.b64encode(content).decode("utf-8")
decoded = base64.b64decode(encoded).decode("utf-8")
print("Base64:", encoded)
print("解码:", decoded)

print("UUID:", uuid.uuid4())
print("安全 token:", secrets.token_hex(16))

MD5: e53024684c9be1dd3f6114ecc8bbdddc
SHA256: 373a23512f9531ad49ec6ad43ecdda58df01e59d9ec5812d601fd05cc53345d3
HMAC 签名: 9f463b6cc493f9fa8da6f3d5a88a55ea1d93a4f5b4450928316c01020ebd70da
Base64: aGVsbG8gcHl0aG9u
解码: hello python
UUID: e269c68e-4b57-4a0a-ab47-e18d2705bd80
安全 token: ab6f4193ee340721a3fb3a61ceb22ad6


# pickle：Python 对象序列化

`pickle` 可以保存 Python 对象，但不要加载不可信来源的 pickle 文件，因为它存在安全风险。

In [27]:
import pickle

model_info = {
    "name": "demo-model",
    "features": ["age", "score"],
    "params": {"alpha": 0.1}
}

pickle_path = OUTPUT_DIR / "model_info.pkl"

with pickle_path.open("wb") as f:
    pickle.dump(model_info, f)

with pickle_path.open("rb") as f:
    loaded = pickle.load(f)

print(loaded)

{'name': 'demo-model', 'features': ['age', 'score'], 'params': {'alpha': 0.1}}


# argparse：命令行参数

在脚本项目中，`argparse` 可以让程序支持命令行参数。Notebook 中可以先写入一个 `.py` 文件，再用子进程运行。

In [28]:
script_path = BASE_DIR / "cli_demo.py"

script_path.write_text(
    '''
import argparse

parser = argparse.ArgumentParser(description="命令行参数示例")
parser.add_argument("--name", required=True, help="用户名")
parser.add_argument("--repeat", type=int, default=1, help="重复次数")

args = parser.parse_args()

for _ in range(args.repeat):
    print(f"Hello, {args.name}")
'''.strip(),
    encoding="utf-8"
)

result = subprocess.run(
    [sys.executable, str(script_path), "--name", "Alice", "--repeat", "3"],
    capture_output=True,
    text=True
)

print(result.stdout)

Hello, Alice
Hello, Alice
Hello, Alice



# unittest：单元测试

单元测试可以让你确认函数行为是否符合预期，是工程化开发的重要基础。

In [29]:
test_script = BASE_DIR / "test_utils.py"

test_script.write_text(
    '''
import unittest

def add(a, b):
    return a + b

def safe_divide(a, b):
    if b == 0:
        raise ValueError("b cannot be zero")
    return a / b

class TestUtils(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(1, 2), 3)

    def test_safe_divide(self):
        self.assertEqual(safe_divide(6, 2), 3)

    def test_safe_divide_zero(self):
        with self.assertRaises(ValueError):
            safe_divide(1, 0)

if __name__ == "__main__":
    unittest.main()
'''.strip(),
    encoding="utf-8"
)

result = subprocess.run(
    [sys.executable, str(test_script)],
    capture_output=True,
    text=True
)

print("stdout:")
print(result.stdout)
print("stderr:")
print(result.stderr)

stdout:

stderr:
...
----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK



# 综合项目：异步日志分析流水线

下面实现一个小型工程化项目：

生成模拟日志文件，异步读取多个日志文件，使用正则表达式解析日志，用 `Counter` 统计接口访问量，用 SQLite 保存结果，用 logging 输出过程日志，最后导出 JSON 报告。

In [30]:
import random
from datetime import datetime, timedelta

api_paths = ["/api/users", "/api/orders", "/api/products", "/api/login"]
levels = ["INFO", "WARNING", "ERROR"]

log_source_dir = DATA_DIR / "app_logs"
log_source_dir.mkdir(exist_ok=True)

base_time = datetime.now()

for file_idx in range(3):
    lines = []
    for i in range(50):
        timestamp = (base_time + timedelta(seconds=i)).strftime("%Y-%m-%d %H:%M:%S")
        level = random.choices(levels, weights=[0.75, 0.15, 0.10])[0]
        path = random.choice(api_paths)
        status = random.choices([200, 201, 400, 401, 404, 500], weights=[60, 10, 10, 5, 10, 5])[0]
        cost_ms = random.randint(10, 1200)
        lines.append(f"{timestamp} | {level} | path={path} | status={status} | cost_ms={cost_ms}\n")

    (log_source_dir / f"app_{file_idx}.log").write_text("".join(lines), encoding="utf-8")

print("日志文件生成完成:")
print(list(log_source_dir.glob("*.log")))

日志文件生成完成:
[PosixPath('python_stdlib_practice/data/app_logs/app_0.log'), PosixPath('python_stdlib_practice/data/app_logs/app_2.log'), PosixPath('python_stdlib_practice/data/app_logs/app_1.log')]


In [31]:
import asyncio
import re
from collections import Counter, defaultdict
import sqlite3
import json
import logging

pipeline_logger = logging.getLogger("log_pipeline")
pipeline_logger.setLevel(logging.INFO)
pipeline_logger.handlers.clear()

pipeline_log_path = LOG_DIR / "pipeline.log"
pipeline_handler = logging.FileHandler(pipeline_log_path, encoding="utf-8")
pipeline_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
pipeline_logger.addHandler(pipeline_handler)

log_pattern = re.compile(
    r"(?P<time>.*?) \| (?P<level>\w+) \| path=(?P<path>\S+) \| status=(?P<status>\d+) \| cost_ms=(?P<cost_ms>\d+)"
)

async def read_file_async(path):
    return await asyncio.to_thread(path.read_text, encoding="utf-8")

async def parse_log_file(path):
    pipeline_logger.info(f"开始解析 {path.name}")
    text = await read_file_async(path)
    records = []

    for line in text.splitlines():
        match = log_pattern.match(line)
        if not match:
            continue

        item = match.groupdict()
        item["status"] = int(item["status"])
        item["cost_ms"] = int(item["cost_ms"])
        item["source_file"] = path.name
        records.append(item)

    pipeline_logger.info(f"完成解析 {path.name}，记录数 {len(records)}")
    return records

async def run_pipeline():
    files = list(log_source_dir.glob("*.log"))
    tasks = [parse_log_file(path) for path in files]
    nested_records = await asyncio.gather(*tasks)
    records = [record for group in nested_records for record in group]
    return records

records = await run_pipeline()
print("总记录数:", len(records))
print("前 3 条:", records[:3])

总记录数: 150
前 3 条: [{'time': '2026-07-07 16:51:32', 'level': 'INFO', 'path': '/api/products', 'status': 201, 'cost_ms': 415, 'source_file': 'app_0.log'}, {'time': '2026-07-07 16:51:33', 'level': 'INFO', 'path': '/api/orders', 'status': 201, 'cost_ms': 17, 'source_file': 'app_0.log'}, {'time': '2026-07-07 16:51:34', 'level': 'WARNING', 'path': '/api/login', 'status': 401, 'cost_ms': 1103, 'source_file': 'app_0.log'}]


In [32]:
path_counter = Counter(record["path"] for record in records)
level_counter = Counter(record["level"] for record in records)
status_counter = Counter(record["status"] for record in records)

cost_by_path = defaultdict(list)
for record in records:
    cost_by_path[record["path"]].append(record["cost_ms"])

avg_cost_by_path = {
    path: round(sum(costs) / len(costs), 2)
    for path, costs in cost_by_path.items()
}

summary = {
    "total_records": len(records),
    "path_count": dict(path_counter),
    "level_count": dict(level_counter),
    "status_count": dict(status_counter),
    "avg_cost_by_path": avg_cost_by_path
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "total_records": 150,
  "path_count": {
    "/api/products": 38,
    "/api/orders": 40,
    "/api/login": 36,
    "/api/users": 36
  },
  "level_count": {
    "INFO": 110,
    "WARNING": 27,
    "ERROR": 13
  },
  "status_count": {
    "201": 24,
    "401": 6,
    "404": 14,
    "200": 85,
    "400": 16,
    "500": 5
  },
  "avg_cost_by_path": {
    "/api/products": 543.32,
    "/api/orders": 583.6,
    "/api/login": 652.61,
    "/api/users": 555.42
  }
}


In [33]:
analysis_db_path = OUTPUT_DIR / "log_analysis.db"

with sqlite3.connect(analysis_db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS log_summary")
    cursor.execute("""
    CREATE TABLE log_summary (
        metric_type TEXT NOT NULL,
        metric_name TEXT NOT NULL,
        metric_value REAL NOT NULL
    )
    """)

    rows_to_insert = []

    for path, count in path_counter.items():
        rows_to_insert.append(("path_count", path, count))

    for level, count in level_counter.items():
        rows_to_insert.append(("level_count", level, count))

    for status, count in status_counter.items():
        rows_to_insert.append(("status_count", str(status), count))

    for path, avg_cost in avg_cost_by_path.items():
        rows_to_insert.append(("avg_cost_by_path", path, avg_cost))

    cursor.executemany(
        "INSERT INTO log_summary (metric_type, metric_name, metric_value) VALUES (?, ?, ?)",
        rows_to_insert
    )
    conn.commit()

with sqlite3.connect(analysis_db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM log_summary LIMIT 10")
    print(cursor.fetchall())

report_path = OUTPUT_DIR / "log_report.json"
report_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("数据库:", analysis_db_path)
print("报告:", report_path)
print("流水线日志:", pipeline_log_path)

[('path_count', '/api/products', 38.0), ('path_count', '/api/orders', 40.0), ('path_count', '/api/login', 36.0), ('path_count', '/api/users', 36.0), ('level_count', 'INFO', 110.0), ('level_count', 'WARNING', 27.0), ('level_count', 'ERROR', 13.0), ('status_count', '201', 24.0), ('status_count', '401', 6.0), ('status_count', '404', 14.0)]
数据库: python_stdlib_practice/output/log_analysis.db
报告: python_stdlib_practice/output/log_report.json
流水线日志: python_stdlib_practice/logs/pipeline.log
